# 03 — Train the Legal-Metrology NER model (GPU, PackSure)

Primary base: **ModernBERT-base** (`answerdotai/ModernBERT-base`).
Benchmark alternative: **DeBERTa-v3-base** (`microsoft/deberta-v3-base`).
The production model is whichever wins on validation/test metrics AND meets
inference needs — never assumed. Everything below is configurable.

> You cannot run this cell block meaningfully until notebook 02 produced the JSONL splits from YOUR annotated data.

In [ ]:
# GPU check + installs
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - Runtime > Change runtime type > T4 GPU')
!pip install -q "transformers>=4.48" datasets accelerate seqeval sentencepiece protobuf

In [ ]:
# Config — every hyperparameter explicit and editable.
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/packsure/ml/dataset'
OUTPUT_ROOT = '/content/drive/MyDrive/packsure/ml/runs'

BASE_MODEL = 'answerdotai/ModernBERT-base'   # try 'microsoft/deberta-v3-base' for the benchmark run
RUN_NAME = 'modernbert-legal-ner-v1'
HYPER = dict(
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=12,
    weight_decay=0.01,
    warmup_ratio=0.1,
    max_seq_length=192,          # labels are short lines; 192 tokens is plenty
    seed=42,
    early_stopping_patience=3,
)
print(BASE_MODEL, HYPER)

In [ ]:
# Load JSONL + label mapping; tokenize with offset mapping and align labels.
import json
from datasets import load_dataset
from transformers import AutoTokenizer

label_mapping = json.load(open(f'{DATA_DIR}/label_mapping.json'))
label_list = label_mapping['labels']
label_to_id = label_mapping['label_to_id']
id_to_label = {int(k): v for k, v in label_mapping['id_to_label'].items()}

ds = load_dataset('json', data_files={
    'train': f'{DATA_DIR}/train/train.jsonl',
    'validation': f'{DATA_DIR}/validation/validation.jsonl',
})
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

def tokenize_and_align(batch):
    tokenized = tokenizer(batch['tokens'], is_split_into_words=True, truncation=True, max_length=HYPER['max_seq_length'])
    aligned = []
    for i, labels in enumerate(batch['labels']):
        word_ids = tokenized.word_ids(batch_index=i)
        previous = None
        row = []
        for word_id in word_ids:
            if word_id is None:
                row.append(-100)
            elif word_id != previous:
                row.append(label_to_id[labels[word_id]])
            else:
                # Subword pieces: label only the first piece of each word.
                row.append(-100)
            previous = word_id
        aligned.append(row)
    tokenized['labels'] = aligned
    return tokenized

encoded = ds.map(tokenize_and_align, batched=True, remove_columns=ds['train'].column_names)
print(encoded)

In [ ]:
# seqeval metrics: per-entity precision/recall/F1 (strict BIO).
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

def compute_metrics(eval_prediction):
    predictions, labels = eval_prediction
    predictions = np.argmax(predictions, axis=-1)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label_row) if l != -100]
        for prediction, label_row in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label_row) if l != -100]
        for prediction, label_row in zip(predictions, labels)
    ]
    return {
        'precision': precision_score(true_labels, true_predictions),
        'recall': recall_score(true_labels, true_predictions),
        'f1': f1_score(true_labels, true_predictions),
    }
print('metrics ready')

In [ ]:
# Train with early stopping + best-model checkpointing.
from transformers import (AutoModelForTokenClassification, DataCollatorForTokenClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)

model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL, num_labels=len(label_list), id2label=id_to_label, label2id=label_to_id,
)

args = TrainingArguments(
    output_dir=f'{OUTPUT_ROOT}/{RUN_NAME}',
    learning_rate=HYPER['learning_rate'],
    per_device_train_batch_size=HYPER['per_device_train_batch_size'],
    per_device_eval_batch_size=HYPER['per_device_eval_batch_size'],
    num_train_epochs=HYPER['num_train_epochs'],
    weight_decay=HYPER['weight_decay'],
    warmup_ratio=HYPER['warmup_ratio'],
    seed=HYPER['seed'],
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=25,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded['train'],
    eval_dataset=encoded['validation'],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=HYPER['early_stopping_patience'])],
)
trainer.train()

In [ ]:
# Validation report (per-entity) — the number you select models on.
predictions = trainer.predict(encoded['validation'])
import numpy as np
preds = np.argmax(predictions.predictions, axis=-1)
true_preds = [[label_list[p] for (p, l) in zip(pr, lr) if l != -100] for pr, lr in zip(preds, predictions.label_ids)]
true_labels = [[label_list[l] for (p, l) in zip(pr, lr) if l != -100] for pr, lr in zip(preds, predictions.label_ids)]
print(classification_report(true_labels, true_preds, digits=3))

In [ ]:
# Export the trained artifact for the NLP service (loads this directory directly).
import shutil
export_dir = f'{OUTPUT_ROOT}/{RUN_NAME}/export'
trainer.save_model(export_dir)
tokenizer.save_pretrained(export_dir)
shutil.copy(f'{DATA_DIR}/label_mapping.json', f'{export_dir}/label_mapping.json')

card = {
    'name': 'Legal-Metrology-NER',
    'base_model': BASE_MODEL,
    'version': RUN_NAME,
    'trained_on': 'PackSure annotated OCR dataset (see ml/dataset/README.md)',
    'labels': label_list,
    'note': 'Metrics in ml/runs - NEVER publish numbers that were not measured.',
}
json.dump(card, open(f'{export_dir}/model_card.json', 'w'), indent=2)
print('Exported to', export_dir)
!ls -la {export_dir}

In [ ]:
# Zip for download -> place into nlp-service/model/legal_metrology_ner/ (see README)
!cd {OUTPUT_ROOT}/{RUN_NAME} && zip -r legal_metrology_ner.zip export
from google.colab import files
# files.download(f'{OUTPUT_ROOT}/{RUN_NAME}/legal_metrology_ner.zip')  # uncomment to download
print('Copy legal_metrology_ner.zip contents into nlp-service/model/legal_metrology_ner/')